# Validação de FIDC (`CotaFIDC`), Letras Financeiras (`LetraFinanceira`), Z-Spread ANBIMA e Carteira em Lote

Este caderno documenta a validação dos instrumentos e métricas complementares de crédito privado do módulo `bz_pricing.instruments.credit`:
1. Cotas de FIDC (`CotaFIDC`), com dois ativos por indexador (`DI + Spread`, `% do DI`, `IPCA`, `IGP-M` e `PRÉ`), além da verificação de consistência inversa por `ytm` e `tir`.
2. Letras Financeiras (`LetraFinanceira`) nas modalidades `SENIOR` (*yield to maturity*) e `SUBORDINADA` / `COMPLEMENTAR` (*yield to call*), com extração da matriz aberta de *spreads* indicativos divulgada pela ANBIMA (`ANBIMADataClient.get_letras_financeiras()`).
3. Validação contra PUs oficiais publicados pela ANBIMA em títulos reais de mercado secundário, cálculo do `Z-Spread` oficial ANBIMA (com e sem *gross-up* de IR pela Lei 12.431) e consulta à ETTJ oficial da ANBIMA (`ANBIMADataClient.get_ettj()`).
4. Precificação em lote de carteira completa via núcleo Cython (`price_credit_portfolio`).

Nos dados abertos da ANBIMA, as Letras Financeiras são divulgadas em matrizes de *spreads* por emissor e prazo, enquanto séries individuais de FIDC não possuem arquivo público de PU por cota; por essa razão, os testes de `CotaFIDC` e `LetraFinanceira` verificam aderência matemática e inversão exata (`ytm` e `tir`), e o confronto direto contra PUs oficiais da ANBIMA é apresentado na Seção 3.

In [1]:
import datetime, os, time
if os.path.basename(os.getcwd()) == 'notebooks':
    os.chdir('..')
import pandas as pd
from bz_pricing.curves import CurveManager
from bz_pricing.data import ANBIMADataClient, DebenturesClient
from bz_pricing.instruments import (
    CotaFIDC,
    LetraFinanceira,
    Debenture,
    HazardRateCurve,
    price_credit_portfolio,
)

ref_dt = datetime.date(2026, 7, 1)
anb_cli = ANBIMADataClient()
deb_cli = DebenturesClient()
cm = CurveManager(allow_download=True)

## 1. Cotas de FIDC (`CotaFIDC`) por Indexador e Inversão `ytm` / `tir`

Para cada uma das cinco famílias de remuneração (`DI + Spread`, `% do DI`, `IPCA`, `IGP-M` e `PRÉ`), avaliam-se dois cronogramas com periodicidade mensal e semestral. Em seguida, a partir do `PU` obtido, executa-se o cálculo inverso de `ytm` (taxa na convenção do indexador) e `tir` (taxa interna de retorno efetiva anual em `BUS/252`) para confirmar a consistência numérica da precificação.

In [2]:
fidcs = [
    (CotaFIDC.from_schedule('FIDC-DI-1',   '2026-01-02', '2028-01-03', indexador='DI',    coupon_rate=0.020, coupon_freq='mensal'),    0.024, 1000.0, 1005.0, 0.135),
    (CotaFIDC.from_schedule('FIDC-DI-2',   '2025-07-01', '2029-07-02', indexador='DI',    coupon_rate=0.025, coupon_freq='semestral'), 0.028, 1000.0, 1020.0, 0.135),
    (CotaFIDC.from_schedule('FIDC-PCT-1',  '2026-01-02', '2028-01-03', indexador='% DI',  multiplicador_di=1.08, coupon_freq='mensal'),    1.10,  1000.0, 1004.0, 0.135),
    (CotaFIDC.from_schedule('FIDC-PCT-2',  '2025-07-01', '2028-07-03', indexador='% DI',  multiplicador_di=1.12, coupon_freq='semestral'), 1.15,  1000.0, 1030.0, 0.135),
    (CotaFIDC.from_schedule('FIDC-IPCA-1', '2025-01-15', '2028-01-17', indexador='IPCA',  coupon_rate=0.075, coupon_freq='semestral'), 0.080, 1100.0, None,   None),
    (CotaFIDC.from_schedule('FIDC-IPCA-2', '2025-06-16', '2029-06-15', indexador='IPCA',  coupon_rate=0.082, coupon_freq='mensal'),    0.085, 1060.0, None,   None),
    (CotaFIDC.from_schedule('FIDC-IGPM-1', '2025-01-15', '2028-01-17', indexador='IGP-M', coupon_rate=0.070, coupon_freq='semestral'), 0.075, 1090.0, None,   None),
    (CotaFIDC.from_schedule('FIDC-IGPM-2', '2025-06-16', '2029-06-15', indexador='IGP-M', coupon_rate=0.078, coupon_freq='mensal'),    0.080, 1055.0, None,   None),
    (CotaFIDC.from_schedule('FIDC-PRE-1',  '2026-01-02', '2028-01-03', indexador='PRÉ',   coupon_rate=0.145, coupon_freq='semestral'), 0.148, 1000.0, None,   None),
    (CotaFIDC.from_schedule('FIDC-PRE-2',  '2025-07-01', '2029-07-02', indexador='PRÉ',   coupon_rate=0.150, coupon_freq='mensal'),    0.152, 1000.0, None,   None),
]

fidc_rows = []
for obj, taxa, vna_v, pupar_v, di_c in fidcs:
    m = obj.risk_metrics(ref_dt, taxa, vna=vna_v, pu_par_val=pupar_v, di_curve=di_c)
    ytm_calc = obj.ytm(ref_dt, m['pu'], vna=vna_v, pu_par_val=pupar_v, di_curve=di_c)
    tir_calc = obj.tir(ref_dt, m['pu'], vna=vna_v, pu_par_val=pupar_v, di_curve=di_c)
    fidc_rows.append({
        'ticker': obj.ticker,
        'indexador': obj.indexador,
        'pct_di': obj._is_pct_di(),
        'taxa_entrada': taxa,
        'pu': m['pu'],
        'ytm_inverso': ytm_calc,
        'erro_ytm': abs(ytm_calc - taxa),
        'tir_efetiva': tir_calc,
        'duration_du': m['duration_du'],
        'dv01': m['dv01'],
        'convexity': m['convexity'],
    })

pd.DataFrame(fidc_rows)

,ticker,indexador,pct_di,taxa_entrada,pu,ytm_inverso,erro_ytm,tir_efetiva,duration_du,dv01,convexity
0,FIDC-DI-1,DI,False,0.024,999.711952,0.024,2.873941e-12,0.162240,339.61,0.115921,2.439479
1,FIDC-DI-2,DI,False,0.028,1012.730296,0.028,2.315197e-13,0.166780,616.47,0.212334,6.737424
2,FIDC-PCT-1,% DI,True,1.100,1000.553103,1.100,7.500667e-13,0.149460,342.25,0.118220,2.517184
3,FIDC-PCT-2,% DI,True,1.150,1023.141021,1.150,1.571765e-11,0.156759,443.23,0.155570,3.801520
4,FIDC-IPCA-1,IPCA,False,0.080,1128.707735,0.080,7.319215e-12,0.080000,361.95,0.150111,3.095056
5,FIDC-IPCA-2,IPCA,False,0.085,1055.972919,0.085,1.483619e-12,0.085000,659.35,0.254647,8.488502
6,FIDC-IGPM-1,IGP-M,False,0.075,1116.043152,0.075,3.954587e-12,0.075000,363.56,0.149781,3.139954
7,FIDC-IGPM-2,IGP-M,False,0.080,1053.314809,0.080,4.824752e-13,0.080000,662.94,0.256573,8.626266
8,FIDC-PRE-1,PRÉ,False,0.148,1064.082671,0.148,7.480683e-12,0.148000,331.22,0.121831,2.448169
9,FIDC-PRE-2,PRÉ,False,0.152,995.754860,0.152,6.635886e-12,0.152000,616.79,0.211563,6.953185


## 2. Letras Financeiras (`LetraFinanceira`) e Matriz Oficial de Spreads ANBIMA

Nesta etapa, consultam-se os *spreads* indicativos oficiais divulgados pela ANBIMA para o emissor `ABC` nos vértices de 12 e 24 meses (`indicative_spread_from_anbima`). Na tabela de precificação, títulos `SENIOR` são descontados até o vencimento contratual (*yield to maturity*), enquanto títulos `SUBORDINADA` e `COMPLEMENTAR` têm o horizonte truncado na `call_date` (*yield to call*), com verificação da inversão por `ytm` e `tir`.

In [3]:
lf_raw = anb_cli.get_letras_financeiras()
print(f"Última referência na matriz LF ANBIMA: {lf_raw['ultimas_referencias'][0]}")

spread_abc_12m = LetraFinanceira.indicative_spread_from_anbima('ABC', 12, client=anb_cli)
spread_abc_24m = LetraFinanceira.indicative_spread_from_anbima('ABC', 24, client=anb_cli)
print(f'Spread indicativo ABC ANBIMA: 12M={spread_abc_12m*100:.4f}% | 24M={spread_abc_24m*100:.4f}%')

lfs = [
    (LetraFinanceira.from_bullet('LF-SEN-DI',      '2026-01-02', '2028-01-03', indexador='DI',   coupon_rate=spread_abc_24m, nominal_value=50000.0, classe='SENIOR', emissor='ABC'), spread_abc_24m, 50000.0, 51500.0, 0.135),
    (LetraFinanceira.from_bullet('LF-SEN-PRE',     '2026-01-02', '2028-01-03', indexador='PRÉ',  coupon_rate=0.140,          nominal_value=50000.0, classe='SENIOR'), 0.142, 50000.0, None, None),
    (LetraFinanceira.from_bullet('LFSN-SUB-DI',    '2026-01-02', '2033-01-03', indexador='DI',   coupon_rate=0.012,          nominal_value=50000.0, classe='SUBORDINADA', call_date='2031-01-02'), 0.014, 50000.0, 51500.0, 0.135),
    (LetraFinanceira.from_bullet('LFSC-COMP-IPCA', '2025-01-15', '2032-01-15', indexador='IPCA', coupon_rate=0.072,          nominal_value=50000.0, classe='COMPLEMENTAR', call_date='2030-01-15'), 0.075, 54000.0, None, None),
]

lf_rows = []
for obj, taxa, vna_v, pupar_v, di_c in lfs:
    m = obj.risk_metrics(ref_dt, taxa, vna=vna_v, pu_par_val=pupar_v, di_curve=di_c)
    ytm_calc = obj.ytm(ref_dt, m['pu'], vna=vna_v, pu_par_val=pupar_v, di_curve=di_c)
    tir_calc = obj.tir(ref_dt, m['pu'], vna=vna_v, pu_par_val=pupar_v, di_curve=di_c)
    lf_rows.append({
        'ticker': obj.ticker,
        'classe': obj.classe,
        'horizonte': obj.maturity_date,
        'taxa_entrada': taxa,
        'pu': m['pu'],
        'ytm_inverso': ytm_calc,
        'tir_efetiva': tir_calc,
        'duration_du': m['duration_du'],
        'dv01': m['dv01'],
    })

pd.DataFrame(lf_rows)

Última referência na matriz LF ANBIMA: 2026-11-12
Spread indicativo ABC ANBIMA: 12M=0.2484% | 24M=0.3194%


,ticker,classe,horizonte,taxa_entrada,pu,ytm_inverso,tir_efetiva,duration_du,dv01
0,LF-SEN-DI,SENIOR,2028-01-03,0.003194,51500.000000,0.003194,0.138625,378.0,6.784498
1,LF-SEN-PRE,SENIOR,2028-01-03,0.142000,53134.585809,0.142000,0.142000,378.0,6.979149
2,LFSN-SUB-DI,SUBORDINADA,2031-01-02,0.014000,51047.274070,0.014000,0.150890,1127.0,19.836366
3,LFSC-COMP-IPCA,COMPLEMENTAR,2030-01-15,0.075000,59138.562228,0.075000,0.075000,884.0,19.298076


## 3. Validação contra PU Oficial ANBIMA, ETTJ, Z-Spread (com e sem *Gross-Up*) e *Hazard Rate*

Esta seção reúne quatro verificações sobre a metodologia oficial da ANBIMA e o modelo estrutural de risco de crédito:

1. Leitura da Estrutura a Termo das Taxas de Juros (`ETTJ`) oficial da ANBIMA via `ANBIMADataClient.get_ettj()`.
2. Confronto entre o `PU` calculado pela biblioteca e o `PU` oficial divulgado pela ANBIMA no arquivo `db261030.txt` (data-base `30/10/2026`) para debêntures reais (`VALEA0` em IPCA e `AALM12` em DI + Spread), acompanhado de `ytm`, `tir` e `Z-Spread`.
3. Reprodução do exemplo de referência de `Z-Spread` em título IPCA não incentivado (`i = 7,3348%`, taxa livre de risco `6,4904%`, resultado `0,7929%`) e da versão incentivada com *gross-up* de alíquota regressiva de IR.
4. Precificação ajustada ao risco de crédito com `HazardRateCurve.from_credit_spreads`, cálculo de CVA, inversão do *hazard rate* implícito (`implied_hazard_rate`) e `credit_dv01`.


In [4]:
df_ettj = anb_cli.get_ettj()
print('ETTJ ANBIMA (primeiras linhas):')
display(df_ettj.head(5))

# Confronto contra PU oficial ANBIMA em debêntures reais (30/10/2026)
base_anb = datetime.date(2026, 10, 30)
df_sec = deb_cli.get_market_prices(base_anb)
curva_pre_anb = cm.get_curve('PRE', base_anb)

rows_of = []
for tk, is_inc, rf_rate, vna_of in [('VALEA0', True, 0.065, 1096.791130), ('AALM12', False, None, None)]:
    row_of = df_sec[df_sec['codigo'] == tk].iloc[0]
    pu_of = float(row_of['pu_mercado'])
    taxa_ind = float(row_of['taxa_indicativa']) / 100.0
    deb = Debenture.from_ticker(tk, client=deb_cli)
    di_c = curva_pre_anb if deb._is_di_linked() else None
    m = deb.risk_metrics(base_anb, taxa_ind, vna=vna_of, di_curve=di_c, risk_free_rate=rf_rate, incentivada=is_inc)
    ytm_of = deb.ytm(base_anb, pu_of, vna=vna_of, di_curve=di_c)
    tir_of = deb.tir(base_anb, pu_of, vna=vna_of, di_curve=di_c)
    rows_of.append({
        'ticker': tk,
        'indexador': deb.indexador,
        'pu_calc': m['pu'],
        'pu_anbima': pu_of,
        'diff_pu': m['pu'] - pu_of,
        'taxa_anbima_%': taxa_ind * 100.0,
        'ytm_do_pu_of_%': ytm_of * 100.0,
        'tir_do_pu_of_%': tir_of * 100.0,
        'z_spread_%': m.get('z_spread', float('nan')) * 100.0,
    })

print('Confronto direto contra PU oficial ANBIMA (30/10/2026):')
display(pd.DataFrame(rows_of))

# Exemplo analítico de Z-Spread ANBIMA (com e sem gross-up de IR)
ativo_z = CotaFIDC.from_schedule('ZSPREAD-IPCA', '2025-01-15', '2030-01-15', indexador='IPCA', coupon_rate=0.065)
m_nao_inc = ativo_z.risk_metrics(ref_dt, 0.073348, vna=1080.0, risk_free_rate=0.064904, incentivada=False)
m_inc     = ativo_z.risk_metrics(ref_dt, 0.073348, vna=1080.0, risk_free_rate=0.064904, incentivada=True)
print(f"Z-Spread não incentivada: {m_nao_inc['z_spread']*100:.4f}% (referência: 0.7929%)")
print(f"Z-Spread incentivada (gross-up): {m_inc['z_spread']*100:.4f}% | Taxa gross-up: {m_inc['taxa_gross_up']*100:.4f}%")

# Modelo estrutural de Hazard Rate, CVA e Credit DV01
hz_fidc = HazardRateCurve.from_credit_spreads([252, 504, 1260], [0.008, 0.010, 0.012], recovery_rate=0.40)
rk_fidc = ativo_z.risky_present_value(ref_dt, 0.064904, hazard_curve=hz_fidc, vna=1080.0)
lam_fidc = ativo_z.implied_hazard_rate(ref_dt, market_pu=rk_fidc['pv_risky'], risk_free_rate_or_curve=0.064904, recovery_rate=0.40, vna=1080.0)
cdv_fidc = ativo_z.credit_dv01(ref_dt, 0.064904, hazard_curve=hz_fidc, vna=1080.0)
print(f"FIDC-IPCA Risco de Crédito -> PV Riscado: R$ {rk_fidc['pv_risky']:.4f} | CVA: R$ {rk_fidc['cva']:.4f} | Hazard Implícito: {lam_fidc*100:.4f}% a.a. | Credit DV01: R$ {cdv_fidc:.6f}")


ETTJ ANBIMA (primeiras linhas):


,du,ettj_ipca,ettj_pref,inflacao_implicita
0,126,0.042411,0.137985,0.091685
1,252,0.060330,0.141224,0.076291
2,378,0.069118,0.144288,0.070310
3,504,0.073441,0.146785,0.068326
4,630,0.075576,0.148714,0.067998


Confronto direto contra PU oficial ANBIMA (30/10/2026):


,ticker,indexador,pu_calc,pu_anbima,diff_pu,taxa_anbima_%,ytm_do_pu_of_%,tir_do_pu_of_%,z_spread_%
0,VALEA0,IPCA,1022.352694,1022.352694,0.000000,7.6229,7.622900,7.622900,2.333008
1,AALM12,DI,1025.331425,1025.332013,-0.000588,0.8877,0.887671,15.995773,0.887700


Z-Spread não incentivada: 0.7929% (referência: 0.7929%)
Z-Spread incentivada (gross-up): 2.2092% | Taxa gross-up: 8.8430%
FIDC-IPCA Risco de Crédito -> PV Riscado: R$ 1049.9063 | CVA: R$ 33.6445 | Hazard Implícito: 1.6890% a.a. | Credit DV01: R$ 0.193484


## 4. Benchmark de Carteira em Lote via Cython (`price_credit_portfolio`)

Executa-se a precificação conjunta de 200 posições heterogêneas (FIDC `DI + Spread` e título `IPCA` incentivado com cálculo simultâneo de `Z-Spread`, `duration`, `DV01` e `convexity`) através do núcleo compilado em Cython (`price_credit_portfolio`).

In [5]:
positions = [
    {'instrument': fidcs[0][0], 'discount_rate': 0.024,    'quantity': 50,  'vna': 1000.0, 'pu_par_val': 1005.0, 'di_curve': 0.135},
    {'instrument': ativo_z,     'discount_rate': 0.073348, 'quantity': 100, 'vna': 1080.0, 'risk_free_rate': 0.064904, 'incentivada': True},
] * 100

t0 = time.perf_counter()
df_port = price_credit_portfolio(positions, ref_dt)
elapsed = time.perf_counter() - t0
print(f'Carteira de {len(df_port)} ativos precificada em {elapsed:.4f} s')
df_port.head(4)

Carteira de 200 ativos precificada em 2.5024 s


,ticker,classe,indexador,quantity,pu_par,pu,valor_financeiro,cotacao,duration_du,modified_duration,dv01,dv01_financeiro,convexity,z_spread
0,FIDC-DI-1,CotaFIDC,DI,50.0,1005.000000,999.711952,49985.59,99.4738,339.61,1.159541,0.115921,5.796036,2.439479,0.024000
1,ZSPREAD-IPCA,CotaFIDC,IPCA,100.0,1083.243565,1056.980187,105698.01,97.5754,791.42,2.925954,0.309268,30.926758,11.932196,0.022092
2,FIDC-DI-1,CotaFIDC,DI,50.0,1005.000000,999.711952,49985.59,99.4738,339.61,1.159541,0.115921,5.796036,2.439479,0.024000
3,ZSPREAD-IPCA,CotaFIDC,IPCA,100.0,1083.243565,1056.980187,105698.01,97.5754,791.42,2.925954,0.309268,30.926758,11.932196,0.022092


## 5. Referências Normativas e Bibliográficas

- **ANBIMA – Associação Brasileira das Entidades dos Mercados Financeiro e de Capitais.** *Metodologias ANBIMA de Precificação — Fundos de Investimento em Direitos Creditórios (FIDC)*. Rio de Janeiro: ANBIMA. Disponível em: <https://www.anbima.com.br/pt_br/informar/precos-e-indices/precos/fidc.htm>.
- **ANBIMA – Associação Brasileira das Entidades dos Mercados Financeiro e de Capitais.** *Metodologias de Precificação — Letras Financeiras*. Rio de Janeiro: ANBIMA. Disponível em: <https://www.anbima.com.br/pt_br/informar/precos-e-indices/precos/letras-financeiras.htm>.
- **ANBIMA – Associação Brasileira das Entidades dos Mercados Financeiro e de Capitais.** *Metodologia ANBIMA de Cálculo dos Spreads de Crédito (Z-Spread)*. Rio de Janeiro: ANBIMA. Disponível em: <https://www.anbima.com.br/pt_br/informar/precos-e-indices/precos/>.
- **ANBIMA – Associação Brasileira das Entidades dos Mercados Financeiro e de Capitais.** *Dados Abertos ANBIMA — Preços e Taxas Indicativas (Letras Financeiras, FIDC e ETTJ)*. Disponível em: <https://data.anbima.com.br/>.